# 📦 Notebook 01 — Data Loading & Schema Validation
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the PubMedQA dataset from HuggingFace
- Validate schema: confirm `instruction`, `input`, `output` columns exist
- Log raw row count, null counts, and a 5-row sample
- Save raw data locally as `data/raw/pubmedqa_raw.csv`
- Generate `reports/schema_validation_report.md`

### 📋 Deliverables
- `data/raw/pubmedqa_raw.csv`
- `reports/schema_validation_report.md`

---

## 1. Imports & Configuration

In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DATA_DIR = '../data/raw'
REPORTS_DIR = '../reports'
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(RAW_DATA_DIR, 'pubmedqa_raw.csv')
REPORT_PATH = os.path.join(REPORTS_DIR, 'schema_validation_report.md')

# ── Dataset config ─────────────────────────────────────────────────────────
DATASET_NAME = 'llamafactory/PubMedQA'
EXPECTED_COLUMNS = ['instruction', 'input', 'output']
EXPECTED_ROW_COUNT = 10_000

print('✅ Imports successful')
print(f'📁 Raw data → {OUTPUT_PATH}')
print(f'📄 Report  → {REPORT_PATH}')

✅ Imports successful
📁 Raw data → ../data/raw\pubmedqa_raw.csv
📄 Report  → ../reports\schema_validation_report.md


## 2. Load Dataset from HuggingFace

In [2]:
print(f'⏳ Loading dataset: {DATASET_NAME} ...')

dataset = load_dataset(DATASET_NAME)

print('\n✅ Dataset loaded successfully!')
print('\n📊 Dataset overview:')
print(dataset)

⏳ Loading dataset: llamafactory/PubMedQA ...

✅ Dataset loaded successfully!

📊 Dataset overview:
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1000
    })
})


## 3. Inspect Available Splits

In [3]:
print('🔍 Available splits:', list(dataset.keys()))
print()

for split_name, split_data in dataset.items():
    print(f'  Split: "{split_name}" → {len(split_data):,} records')

🔍 Available splits: ['train', 'test']

  Split: "train" → 10,000 records
  Split: "test" → 1,000 records


## 4. Convert to Pandas DataFrame

In [4]:
main_split = list(dataset.keys())[0]
print(f'📌 Using split: "{main_split}"')

df = pd.DataFrame(dataset[main_split])

print(f'\n✅ Converted to DataFrame')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

📌 Using split: "train"

✅ Converted to DataFrame
   Shape: 10,000 rows × 3 columns


## 5. Schema Validation

In [5]:
print('🔍 Schema Validation')
print('─' * 50)

# Check columns
actual_columns = df.columns.tolist()
missing_cols = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_cols = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

print(f'Expected columns: {EXPECTED_COLUMNS}')
print(f'Actual columns:   {actual_columns}')
print(f'Missing columns:  {missing_cols if missing_cols else "None"}')
print(f'Extra columns:    {extra_cols if extra_cols else "None"}')

if not missing_cols and not extra_cols:
    schema_status = 'PASSED — all 3 expected columns present, no drift'
    print(f'\n✅ {schema_status}')
else:
    schema_status = f'FAILED — Missing: {missing_cols}, Extra: {extra_cols}'
    print(f'\n❌ {schema_status}')

# Check row count
print(f'\nExpected ~{EXPECTED_ROW_COUNT:,} rows, got {len(df):,} rows')
if len(df) >= EXPECTED_ROW_COUNT:
    print('✅ Row count OK')
else:
    print(f'⚠️  Fewer rows than expected')

# Check data types
print(f'\n📋 Data Types:')
print(df.dtypes)

🔍 Schema Validation
──────────────────────────────────────────────────
Expected columns: ['instruction', 'input', 'output']
Actual columns:   ['instruction', 'input', 'output']
Missing columns:  None
Extra columns:    None

✅ PASSED — all 3 expected columns present, no drift

Expected ~10,000 rows, got 10,000 rows
✅ Row count OK

📋 Data Types:
instruction    object
input          object
output         object
dtype: object


## 6. Null Counts

In [6]:
print('🔎 Missing Values per Column:')
print('─' * 40)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df)
print()

if missing.sum() == 0:
    null_status = 'No missing values found'
    print(f'✅ {null_status}')
else:
    null_status = f'{missing.sum():,} total missing values'
    print(f'⚠️  {null_status}')

🔎 Missing Values per Column:
────────────────────────────────────────
             Missing Count  Missing %
instruction              0        0.0
input                    0        0.0
output                   0        0.0

✅ No missing values found


## 7. Duplicate Check

In [7]:
n_duplicates = df.duplicated().sum()
print(f'🔎 Duplicate rows: {n_duplicates:,}')

if n_duplicates == 0:
    dup_status = 'No duplicates found'
    print(f'✅ {dup_status}')
else:
    dup_status = f'{n_duplicates} duplicate rows found'
    print(f'⚠️  {dup_status}')

🔎 Duplicate rows: 0
✅ No duplicates found


## 8. 5-Row Sample

In [8]:
print('👀 First 5 records:')
pd.set_option('display.max_colwidth', 150)
df.head(5)


👀 First 5 records:


,instruction,input,output
0,"Answer the question based on the following context: Although the use of alternative medicine in the United States is increasing, no published stud...",Question: Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?,Naturopathy appears to be an effective alternative for relief of specific menopausal symptoms compared to conventional therapy.
1,"Answer the question based on the following context: To estimate the feasibility, utility and resource implications of electronically captured rout...",Question: Can randomised trials rely on existing electronic data?,"Routine data have the potential to support health technology assessment by RCTs. The cost of data collection and analysis is likely to fall, altho..."
2,Answer the question based on the following context: To compare morbidity in two groups of patients who underwent retropubic or laparoscopic radica...,Question: Is laparoscopic radical prostatectomy better than traditional retropubic radical prostatectomy?,The results of our non-randomized study show that up to now laparoscopic radical prostatectomy does not provide significant advantages in terms of...
3,Answer the question based on the following context: Irritable bowel syndrome (IBS) might develop after gastroenteritis. Most previous studies of t...,Question: Does bacterial gastroenteritis predispose people to functional gastrointestinal disorders?,"Symptoms consistent with IBS and functional diarrhea occur more frequently in people after bacterial gastroenteritis compared with controls, even ..."
4,Answer the question based on the following context: Urgent colonoscopy has been proposed for the diagnosis and management of acute colonic diverti...,Question: Is early colonoscopy after admission for acute diverticular bleeding needed?,No significant association is apparent between the timing of colonoscopy after admission and encountering active bleeding or nonbleeding stigmata....


## 9. Basic Text Statistics

In [9]:
text_cols = df.select_dtypes(include='object').columns.tolist()

print('📏 Text Length Statistics (character count):')
print('─' * 50)

for col in text_cols:
    lengths = df[col].astype(str).str.len()
    print(f'\n[{col}]')
    print(f'  Min:    {lengths.min():,}')
    print(f'  Max:    {lengths.max():,}')
    print(f'  Mean:   {lengths.mean():,.0f}')
    print(f'  Median: {lengths.median():,.0f}')

📏 Text Length Statistics (character count):
──────────────────────────────────────────────────

[instruction]
  Min:    240
  Max:    3,982
  Mean:   1,388
  Median: 1,373

[input]
  Min:    21
  Max:    339
  Mean:   107
  Median: 104

[output]
  Min:    35
  Max:    2,114
  Mean:   287
  Median: 264


## 10. Save Raw Data

In [10]:
df.to_csv(OUTPUT_PATH, index=False)

file_size = os.path.getsize(OUTPUT_PATH) / 1024**2

print(f'✅ Raw dataset saved to: {OUTPUT_PATH}')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.shape[1]}')
print(f'   File size: {file_size:.1f} MB')

✅ Raw dataset saved to: ../data/raw\pubmedqa_raw.csv
   Rows: 10,000
   Columns: 3
   File size: 17.1 MB


## 11. Generate Schema Validation Report

In [11]:
sample_md = df.head(5).to_markdown(index=False)

report = f"""# Schema Validation Report
**Healthcare RAG-Powered Medical Q&A Assistant**

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Dataset Info
| Item | Value |
|---|---|
| Source | `{DATASET_NAME}` |
| Split | `{main_split}` |
| Row count | {len(df):,} |
| Expected rows | ~{EXPECTED_ROW_COUNT:,} |

## Schema Validation
| Item | Value |
|---|---|
| Expected columns | {EXPECTED_COLUMNS} |
| Actual columns | {actual_columns} |
| Missing columns | {missing_cols if missing_cols else 'None'} |
| Extra columns | {extra_cols if extra_cols else 'None'} |
| **Status** | **{schema_status}** |

## Null Counts
{missing_df.to_markdown()}

**Status:** {null_status}

## Duplicates
| Item | Value |
|---|---|
| Duplicate rows | {n_duplicates} |
| **Status** | **{dup_status}** |

## 5-Row Sample
{sample_md}
"""

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(f'✅ Schema validation report saved to: {REPORT_PATH}')

✅ Schema validation report saved to: ../reports\schema_validation_report.md


## ✅ Summary

| Item | Result |
|---|---|
| Dataset | `llamafactory/PubMedQA` |
| Records | {len(df):,} |
| Columns | {actual_columns} |
| Schema | {schema_status} |
| Missing values | {null_status} |
| Duplicates | {dup_status} |
| Saved raw CSV | `data/raw/pubmedqa_raw.csv` |
| Saved report | `reports/schema_validation_report.md` |

---

### ➡️ Next Step
Open **`02_preprocessing.ipynb`** to clean and normalise the dataset.